In [ ]:
!pip install -q x-transformers

In [ ]:
# ==========================================
# 1. SETUP & MODEL LOADING (FIXED)
# ==========================================
import os
import sys
from huggingface_hub import hf_hub_download

# --- CRITICAL FIX: Download the Model Definition FIRST ---
REPO_ID = "prism-lab/prism-shimmer-100k"
filename = "modeling_prism_gated.py"

print(f"⬇️ Downloading {filename} from Hugging Face...")
if not os.path.exists(filename):
    hf_hub_download(repo_id=REPO_ID, filename=filename, local_dir=".", force_download=True)

# Now that the file exists locally, we can import it
sys.path.append(".") # Ensure current dir is in path
from modeling_prism_gated import PRISMHybrid_RoPE

# Continue with standard imports
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from transformers import AutoTokenizer
import json

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
D_MODEL = 512

print("⏳ Downloading Weights & Config...")
if not os.path.exists("config.json"):
    hf_hub_download(repo_id=REPO_ID, filename="config.json", local_dir=".")
if not os.path.exists("pytorch_model.bin"):
    hf_hub_download(repo_id=REPO_ID, filename="pytorch_model.bin", local_dir=".")

with open("config.json", "r") as f: config = json.load(f)
tokenizer = AutoTokenizer.from_pretrained(REPO_ID)

# Initialize Model
model = PRISMHybrid_RoPE(
    vocab_size=config['vocab_size'], d_model=config['d_model'],
    num_encoder_layers=config['num_encoder_layers'], num_refining_layers=0,
    num_decoder_layers=6, num_heads=8, dff=2048, max_length=128, dropout=0.0
).to(DEVICE)

model.load_state_dict(torch.load("pytorch_model.bin", map_location=DEVICE))
model.eval()
print("✅ Model Loaded Successfully.")

In [ ]:
# @title 🧭 Extended Phase Compass: Synonyms vs Antonyms vs Randoms
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from transformers import AutoTokenizer
from huggingface_hub import hf_hub_download
import os
import json


# ==========================================
# 2. DEFINING THE CANDIDATE PAIRS
# ==========================================
# Master list of all candidates
candidates_raw = [
    # --- ORIGINAL LIST ---
    ("Euro", "Geld", "Synonym"),
    ("Auto", "Wagen", "Synonym"),
    ("schnell", "rasch", "Synonym"),
    ("Stimme", "Wahl", "Synonym"),
    ("Zeit", "Uhr", "Synonym"),
    ("Start", "Beginn", "Synonym"),
    ("Ende", "Schluss", "Synonym"),
    ("Raum", "Platz", "Synonym"),

    # --- NEW GERMAN SYNONYMS ---
    ("Haus", "Heim", "Synonym"),
    ("Boot", "Schiff", "Synonym"),
    ("See", "Meer", "Synonym"),
    ("Wald", "Forst", "Synonym"),
    ("Weg", "Pfad", "Synonym"),
    ("Berg", "Gipfel", "Synonym"),
    ("Mund", "Maul", "Synonym"),
    ("Pferd", "Ross", "Synonym"),
    ("Hund", "Tier", "Synonym"),
    ("Reise", "Fahrt", "Synonym"),
    ("Angst", "Furcht", "Synonym"),
    ("Mut", "Traute", "Synonym"),
    ("Glück", "Dusel", "Synonym"),
    ("Ding", "Sache", "Synonym"),
    ("Welt", "Erde", "Synonym"),
    ("Stadt", "Ort", "Synonym"),
    ("Vater", "Papa", "Synonym"),
    ("Mutter", "Mama", "Synonym"),
    ("klug", "weise", "Synonym"),
    ("klug", "schlau", "Synonym"),
    ("schön", "hübsch", "Synonym"),
    ("klein", "winzig", "Synonym"),
    ("stark", "fest", "Synonym"),
    ("neu", "frisch", "Synonym"),
    ("still", "leise", "Synonym"),
    ("froh", "heiter", "Synonym"),
    ("dunkel", "finster", "Synonym"),
    ("kalt", "eisig", "Synonym"),
    ("rennen", "laufen", "Synonym"),
    ("reden", "sagen", "Synonym"),
    ("sehen", "schauen", "Synonym"),
    ("gehen", "wandern", "Synonym"),
    ("essen", "speisen", "Synonym"),
    ("Wut", "Zorn", "Synonym"),
    ("Dreck", "Schmutz", "Synonym"),
    ("Chance", "Möglichkeit", "Synonym"), # Möglichkeit might be split, but code will check
    ("Lehrer", "Pauker", "Synonym"),
    ("Gott", "Herr", "Synonym"),
    ("Chef", "Boss", "Synonym"),
    ("Haut", "Fell", "Synonym"),
    ("Tor", "Tür", "Synonym"),
    ("Zimmer", "Raum", "Synonym"),
    ("Bahn", "Zug", "Synonym"),
    ("Boot", "Kahn", "Synonym"),
    ("Hose", "Jeans", "Synonym"),
    ("Witz", "Scherz", "Synonym"),
    ("Hass", "Abscheu", "Synonym"),
    ("Fett", "Dick", "Synonym"),
    ("klug", "gescheit", "Synonym"),
    ("dumm", "doof", "Synonym"),
    ("rasch", "flink", "Synonym"),
    ("stumm", "still", "Synonym"),
    ("echt", "wahr", "Synonym"),
    ("korrekt", "richtig", "Synonym"),
    # --- NEW ANTONYMS (OPPOSITES) ---
    ("gut", "böse", "Antonym"),
    ("groß", "klein", "Antonym"),
    ("heiß", "kalt", "Antonym"),
    ("Tag", "Nacht", "Antonym"),
    ("hoch", "tief", "Antonym"),
    ("jung", "alt", "Antonym"),
    ("voll", "leer", "Antonym"),
    ("Liebe", "Hass", "Antonym"),
    ("Licht", "Schatten", "Antonym"),
    ("Start", "Ziel", "Antonym"),
    ("Frage", "Antwort", "Antonym"),
# --- ADDITIONAL ANTONYMS (Fundamental Opposites) ---
    ("Leben", "Tod", "Antonym"),
    ("Freund", "Feind", "Antonym"),
    ("Krieg", "Frieden", "Antonym"),
    ("Sieg", "Niederlage", "Antonym"),
    ("Gewinn", "Verlust", "Antonym"),
    ("Himmel", "Hölle", "Antonym"),
    ("Junge", "Mädchen", "Antonym"),
    ("Vater", "Mutter", "Antonym"),
    ("Bruder", "Schwester", "Antonym"),
    ("Sommer", "Winter", "Antonym"),
    ("Sonne", "Mond", "Antonym"),
    ("Feuer", "Wasser", "Antonym"),
    ("schwarz", "weiß", "Antonym"),
    ("hart", "weich", "Antonym"),
    ("laut", "leise", "Antonym"),
    ("schnell", "langsam", "Antonym"),
    ("teuer", "billig", "Antonym"),
    ("reich", "arm", "Antonym"),
    ("schwer", "leicht", "Antonym"),
    ("nass", "trocken", "Antonym"),
    ("sauber", "schmutzig", "Antonym"),
    ("klug", "dumm", "Antonym"),
    ("stark", "schwach", "Antonym"),
    ("dick", "dünn", "Antonym"),
    ("breit", "schmal", "Antonym"),
    # --- NEW RANDOM/UNRELATED ---
    ("Mond", "Tisch", "Random"),
    ("Brot", "Wolke", "Random"),
    ("Schuh", "Idee", "Random"),
    ("Baum", "Zahn", "Random"),
    ("Glas", "Löwe", "Random"),
    ("Buch", "Suppe", "Random"),
    ("Wand", "Vogel", "Random"),
    ("Gras", "Auto", "Random"),
    ("Salz", "Musik", "Random"),
    ("Dach", "Fisch", "Random"),
    ("Stein", "Wort", "Random"),
    ("Kopf", "Preis", "Random"),
    ("Hand", "Woche", "Random"),
    ("Euro", "Apfel", "Random"),
    ("Auto", "Idee", "Random"),
    ("schnell", "Haus", "Random"),
    ("Zeit", "Fisch", "Random"),
    ("Start", "Milch", "Random"),
    ("Raum", "Laufen", "Random"),
# --- ADDITIONAL RANDOM PAIRS (Noise Floor) ---
    ("Käse", "Mond", "Random"),
    ("Bier", "Tante", "Random"),
    ("Zahn", "Autobahn", "Random"),
    ("Vogel", "Benzin", "Random"),
    ("Computer", "Blume", "Random"),
    ("Glas", "Schaf", "Random"),
    ("Schuh", "Luft", "Random"),
    ("Kaffee", "Stein", "Random"),
    ("Wand", "Butter", "Random"),
    ("Fenster", "Schmerz", "Random"),
    ("Nase", "Rechnung", "Random"),
    ("Hund", "Lampe", "Random"),
    ("Katze", "Strom", "Random"),
    ("Apfel", "Krieg", "Random"),
    ("Löffel", "Angst", "Random"),
    ("Zucker", "Politik", "Random"),
    ("Salz", "Liebe", "Random"),
    ("Pfeffer", "Auto", "Random"),
    ("Stuhl", "Wolke", "Random"),
]

# ==========================================
# 3. HELPER FUNCTIONS
# ==========================================
def is_single_token(word):
    """Check if word is 1 token in vocabulary."""
    ids = tokenizer.encode(word, add_special_tokens=False)
    return len(ids) == 1, ids[0] if len(ids) == 1 else None

def calculate_coherence(id_a, id_b):
    """Extract phases and calculate Mean Resultant Length (R)."""
    # 1. Get Weights (CPU)
    w = model.harmonic_embedding.complex_embedding.weight.detach().cpu()

    # 2. Form Complex Numbers (Real + i*Imag)
    za = torch.complex(w[id_a, :D_MODEL], w[id_a, D_MODEL:])
    zb = torch.complex(w[id_b, :D_MODEL], w[id_b, D_MODEL:])

    # 3. Phase Difference (Angle between vectors)
    diff = torch.angle(za) - torch.angle(zb)

    # 4. Energy Weighting (Magnitude * Magnitude)
    #    Stronger concepts contribute more to the "Phase Compass"
    weights = torch.abs(za) * torch.abs(zb)

    # 5. Convert to Numpy
    diff_np = diff.numpy()
    weights_np = weights.numpy()

    # 6. Calculate Circular Mean (R)
    #    R ranges from 0 (Random/Cancel) to 1 (Perfect Alignment)
    weighted_complex_diffs = weights_np * np.exp(1j * diff_np)
    mean_vector = np.sum(weighted_complex_diffs) / np.sum(weights_np)

    return np.abs(mean_vector), np.angle(mean_vector), diff_np, weights_np

# ==========================================
# 4. EXECUTE ANALYSIS
# ==========================================
valid_pairs = []
results = []

print(f"\n{'Pair':<25} | {'Type':<10} | {'Status':<15} | {'R (Coherence)'}")
print("-" * 75)

for w1, w2, ptype in candidates_raw:
    s1, id1 = is_single_token(w1)
    s2, id2 = is_single_token(w2)

    if s1 and s2:
        R, angle, diffs, weights = calculate_coherence(id1, id2)
        valid_pairs.append({
            "w1": w1, "w2": w2, "type": ptype,
            "R": R, "angle": angle, "diffs": diffs, "weights": weights
        })
        results.append({"Pair": f"{w1}-{w2}", "Type": ptype, "R": R})
        print(f"{w1}-{w2:<20} | {ptype:<10} | ✅ Valid        | {R:.4f}")
    else:
        # Just logging for info, skipped in analysis
        pass
        # print(f"{w1}-{w2:<20} | {ptype:<10} | ❌ Multi-token  | -")

# ==========================================
# 5. STATISTICS
# ==========================================
df = pd.DataFrame(results)
print("\n📊 AGGREGATE STATS (Mean Resultant Length R):")
print(df.groupby("Type")["R"].describe())

# ==========================================
# 6. VISUALIZATION (GRID 3x3)
# ==========================================
# Select Top 3 from each category to show clearest examples
synonyms = sorted([p for p in valid_pairs if p["type"] == "Synonym"], key=lambda x: x["R"], reverse=True)[:3]
antonyms = sorted([p for p in valid_pairs if p["type"] == "Antonym"], key=lambda x: x["R"], reverse=True)[:3]
randoms  = sorted([p for p in valid_pairs if p["type"] == "Random"],  key=lambda x: x["R"], reverse=False)[:3] # Lowest R for randoms

plot_list = synonyms + antonyms + randoms

if len(plot_list) > 0:
    fig = plt.figure(figsize=(15, 12))
    fig.suptitle("Semantic Phase Compass: Synonyms vs Antonyms vs Randoms", fontsize=16, y=0.98)

    # Colors for categories
    colors = {"Synonym": "red", "Antonym": "purple", "Random": "blue"}

    for i, item in enumerate(plot_list):
        ax = fig.add_subplot(3, 3, i+1, projection='polar')

        ptype = item["type"]
        color = colors[ptype]

        # Weighted Histogram of Phase Differences
        ax.hist(item["diffs"], bins=30, weights=item["weights"], color=color, alpha=0.7, density=True)

        # Mean Vector Arrow (The "Compass Needle")
        # Length of arrow = R (Coherence Strength)
        ax.annotate("", xy=(item["angle"], item["R"]), xytext=(0,0),
                    arrowprops=dict(facecolor='black', width=2, headwidth=10))

        # Styling
        ax.set_title(f"{item['w1']} - {item['w2']}\n{ptype}\nR = {item['R']:.3f}", fontsize=11)
        ax.set_yticklabels([]) # Hide radial labels
        ax.set_xticklabels([]) # Hide angular labels

    plt.tight_layout()
    plt.savefig("phase_compass_extended.png", dpi=300)
    plt.show()
    print("\n📸 Saved plot to 'phase_compass_extended.png'")
else:
    print("⚠️ Not enough valid pairs to generate plot.")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# ... [Keep your Candidate Lists & Helper Functions from before] ...

# ==========================================
# 4. EXECUTE ANALYSIS & SELECT BEST EXAMPLES
# ==========================================
valid_pairs = []
results = []

print(f"🚀 Running Phase Compass on {len(candidates_raw)} pairs...")

for w1, w2, ptype in candidates_raw:
    s1, id1 = is_single_token(w1)
    s2, id2 = is_single_token(w2)

    if s1 and s2:
        R, angle, diffs, weights = calculate_coherence(id1, id2)
        valid_pairs.append({
            "w1": w1, "w2": w2, "type": ptype,
            "R": R, "angle": angle, "diffs": diffs, "weights": weights
        })
        results.append({"Pair": f"{w1}-{w2}", "Type": ptype, "R": R})

# ==========================================
# 5. VISUALIZATION (1x3 STRIP)
# ==========================================
# Select the "Best" example for each category (Highest R for Syn/Ant, Lowest for Random)
best_syn = max([p for p in valid_pairs if p["type"] == "Synonym"], key=lambda x: x["R"])
best_ant = max([p for p in valid_pairs if p["type"] == "Antonym"], key=lambda x: x["R"])
best_rnd = min([p for p in valid_pairs if p["type"] == "Random"],  key=lambda x: x["R"])

plot_list = [best_syn, best_ant, best_rnd]
titles = ["A. Synonyms (High Coherence)", "B. Antonyms (High Coherence)", "C. Unrelated (Random Phase)"]
colors = ["#d62728", "#9467bd", "#7f7f7f"] # Red, Purple, Gray

fig = plt.figure(figsize=(12, 4)) # Wide, Short aspect ratio

for i, item in enumerate(plot_list):
    ax = fig.add_subplot(1, 3, i+1, projection='polar')

    # 1. Circular Histogram (The "Cloud")
    # We use 'weights' to show that high-energy frequencies matter more
    ax.hist(item["diffs"], bins=40, weights=item["weights"], color=colors[i], alpha=0.6, density=True)

    # 2. Mean Resultant Vector (The "Needle")
    # The length of this arrow is the PROOF of phase locking.
    ax.annotate("", xy=(item["angle"], item["R"]), xytext=(0,0),
                arrowprops=dict(facecolor='black', width=1.5, headwidth=8, alpha=0.9))

    # 3. Styling
    ax.set_title(f"{titles[i]}\n'{item['w1']}' - '{item['w2']}'\n$R = {item['R']:.2f}$",
                 fontsize=10, fontweight='bold', pad=10)
    ax.set_yticklabels([]) # Hide radial numbers
    ax.set_xticklabels([]) # Hide degree numbers
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 0.6) # Fix scale for fair comparison

plt.tight_layout()
plt.savefig("fig_compass_1x3.png", dpi=300, bbox_inches='tight')
plt.show()

# ==========================================
# 6. STATISTICAL TABLE OUTPUT
# ==========================================
df = pd.DataFrame(results)
print("\n📊 PHASE LOCKING STATISTICS (Mean Resultant Length R)")
print("="*60)
print(f"{'Category':<15} | {'Mean R':<10} | {'Std Dev':<10} | {'Count'}")
print("-" * 60)
stats = df.groupby("Type")["R"].agg(['mean', 'std', 'count'])
for idx, row in stats.iterrows():
    print(f"{idx:<15} | {row['mean']:.4f}     | {row['std']:.4f}     | {int(row['count'])}")

In [ ]:
# ==========================================
# 0. SETUP & DEPENDENCIES
# ==========================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import gc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from transformers import RobertaTokenizerFast
from huggingface_hub import hf_hub_download
from x_transformers import TransformerWrapper, Encoder

# Global Config
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEQ_LEN = 4096
MAX_VOCAB_SIZE = 32768
TOKENIZER_ID = "prism-lab/wikitext-103-prism-32k-seq4k" # <--- YOUR REPO

print(f"🔥  Initializing Phase Compass Analysis on {DEVICE}")

# ==========================================
# 1. ARCHITECTURE DEFINITIONS
# ==========================================
# (Standard Definitions - Collapsed for brevity)
class ComplexDropout(nn.Module):
    def __init__(self, p=0.0): super().__init__(); self.p = p
    def forward(self, z): return z
class RobustPhaseNorm(nn.Module):
    def __init__(self, d, eps=1e-5): super().__init__(); self.scale = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x): return (x / torch.sqrt((x.abs()**2).mean(-1, keepdim=True) + self.eps)) * self.scale
class ModReLU(nn.Module):
    def __init__(self, f): super().__init__(); self.b = nn.Parameter(torch.zeros(f))
    def forward(self, z): return F.relu(z.abs() + self.b) * (z / (z.abs() + 1e-6))
class ComplexToRealBridge(nn.Module):
    def __init__(self, d): super().__init__(); self.proj = nn.Linear(d*2, d); self.norm = nn.LayerNorm(d)
    def forward(self, x): return self.norm(self.proj(torch.cat([x.real, x.imag], -1)))
class DynamicRoSE(nn.Module):
    def __init__(self, n, d):
        super().__init__(); self.raw_embedding = nn.Embedding(n, d); self.adapter = nn.Linear(d, d*2); self.rotation_predictor = nn.Linear(d, d*2)
        self.register_buffer('freqs', torch.exp(torch.arange(0, d) * -(math.log(10000.0)/d)))
    def forward(self, x):
        real = self.raw_embedding(x); params = self.adapter(real); D = real.shape[-1]
        z = torch.complex(params[...,:D], params[...,D:]); r = self.rotation_predictor(real); rx, ry = r.chunk(2, -1)
        drot = torch.complex(rx/torch.sqrt(rx**2+ry**2+1e-6), ry/torch.sqrt(rx**2+ry**2+1e-6))
        pos = torch.arange(real.shape[1], device=x.device).float()
        srot = torch.polar(torch.ones_like(torch.outer(pos, self.freqs)), torch.outer(pos, self.freqs))
        return (z * srot.unsqueeze(0) * drot), real
class HyenaNeuralFilter(nn.Module):
    def __init__(self, d, max_len=1024, h=64):
        super().__init__(); self.d = d; self.register_buffer("freqs", torch.exp(torch.arange(0, h, 2) * -(math.log(10000.0)/h)))
        self.mlp = nn.Sequential(nn.Linear(h, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(), nn.Linear(h, d*2))
    def forward(self, L, dev):
        t = torch.linspace(0, 1, steps=L, device=dev).unsqueeze(-1)
        emb = torch.cat([torch.sin(t*self.freqs), torch.cos(t*self.freqs)], -1)
        out = self.mlp(emb).view(L, self.d, 2); return torch.complex(out[...,0], out[...,1])
class GatedHarmonicConvolution(nn.Module):
    def __init__(self, d, max_len):
        super().__init__(); self.d=d; self.filter_len=max_len; self.neural_filter = HyenaNeuralFilter(d, max_len)
        self.gate_proj = nn.Linear(d*2, d*2); self.mix_real = nn.Linear(d,d); self.mix_imag = nn.Linear(d,d)
        self.out_real = nn.Linear(d,d); self.out_imag = nn.Linear(d,d); self.activation = ModReLU(d); self.norm = RobustPhaseNorm(d)
        self.dropout = ComplexDropout(0.0)
    def forward(self, x, mask=None):
        res = x; x = self.norm(x); B,L,D = x.shape; eff_L = min(L, self.filter_len)
        h = self.neural_filter(eff_L, x.device).unsqueeze(0)
        xt = torch.fft.ifft(torch.fft.fft(x, n=eff_L, dim=1, norm='ortho') * h, n=eff_L, dim=1, norm='ortho')
        if L > eff_L: xt = F.pad(xt, (0,0,0,L-eff_L));
        else: xt = xt[:, :L, :]
        g = torch.sigmoid(self.gate_proj(torch.cat([x.real, x.imag], -1))); gr, gi = g.chunk(2, -1)
        xg = torch.complex(xt.real*gr, xt.imag*gi); mr, mi = self.mix_real, self.mix_imag
        xm = torch.complex(mr(xg.real)-mi(xg.imag), mr(xg.imag)+mi(xg.real)); xa = self.activation(xm); or_, oi = self.out_real, self.out_imag
        out = torch.complex(or_(xa.real)-oi(xa.imag), or_(xa.imag)+oi(xa.real))
        return self.dropout(out) + res
class PRISMEncoder(nn.Module):
    def __init__(self, l, d, max_l): super().__init__(); self.layers = nn.ModuleList([GatedHarmonicConvolution(d, max_l) for _ in range(l)]); self.final_norm = RobustPhaseNorm(d)
    def forward(self, x):
        for layer in self.layers: x = layer(x)
        return self.final_norm(x)

# --- A. BASELINE (Transformer) ---
class LocalBaseline(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.model = TransformerWrapper(
            num_tokens=vocab_size, max_seq_len=SEQ_LEN, use_abs_pos_emb=False, tie_embedding=True,
            attn_layers=Encoder(dim=512, depth=5, heads=8, rotary_pos_emb=True, attn_flash=True, use_scalenorm=False)
        )
    def forward(self, x): return self.model(x)

# --- B. FNET (Hybrid) ---
class FNetBlock(nn.Module):
    def __init__(self, d, df):
        super().__init__(); self.norm_mix = nn.LayerNorm(d); self.norm_ff = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, df), nn.GELU(), nn.Dropout(0), nn.Linear(df, d), nn.Dropout(0))
    def forward(self, x):
        r = x; x = self.norm_mix(x); x = torch.fft.fftn(x.float(), dim=(-2,-1), norm='ortho').real.to(r.dtype); x = x+r
        r = x; x = self.norm_ff(x); x = self.ff(x); return x+r
class FNetEncoder(nn.Module):
    def __init__(self, depth, d, df): super().__init__(); self.layers = nn.ModuleList([FNetBlock(d, df) for _ in range(depth)]); self.norm_out = nn.LayerNorm(d)
    def forward(self, x):
        for l in self.layers: x = l(x)
        return self.norm_out(x)
class HybridFNetMLM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, 512); self.pos_emb = nn.Parameter(torch.zeros(1, SEQ_LEN, 512))
        self.fnet_encoder = FNetEncoder(6, 512, 2048)
        self.transformer_cap = Encoder(dim=512, depth=1, heads=8, rotary_pos_emb=True, attn_flash=True)
        self.final_norm = nn.LayerNorm(512); self.to_logits = nn.Linear(512, vocab_size)
        self.to_logits.weight = self.token_emb.weight # Tie
    def forward(self, x):
        h = self.token_emb(x) + self.pos_emb[:, :x.shape[1], :]
        return self.to_logits(self.final_norm(self.transformer_cap(self.fnet_encoder(h))))

# --- C. PRISM (Phase Coder) ---
class LocalPRISM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.rose = DynamicRoSE(vocab_size, 512); self.prism_encoder = PRISMEncoder(5, 512, SEQ_LEN)
        self.bridge = ComplexToRealBridge(512); self.periscope_proj = nn.Sequential(nn.Linear(1024, 512), nn.LayerNorm(512), nn.GELU())
        self.refiner = Encoder(dim=512, depth=1, heads=8, rotary_pos_emb=True, attn_flash=True)
        self.lm_head = nn.Linear(512, vocab_size); self.lm_head.weight = self.rose.raw_embedding.weight # Tie
    def forward(self, x):
        w, p = self.rose(x); w = self.bridge(self.prism_encoder(w))
        return self.lm_head(self.refiner(self.periscope_proj(torch.cat([w, p], -1))))

# --- D. PILLARS (Split-Stream) ---
class LocalPillars(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.rose = DynamicRoSE(vocab_size, 512); self.particle_down = nn.Linear(512, 256); self.wave_down = nn.Linear(1024, 512)
        self.fnet_pos = nn.Embedding(SEQ_LEN, 256); self.stream_rate = FNetEncoder(9, 256, 1024)
        self.stream_phase = PRISMEncoder(9, 256, SEQ_LEN); self.phase_bridge = ComplexToRealBridge(256)
        self.fusion_proj = nn.Linear(512, 512); self.fusion_norm = nn.LayerNorm(512)
        self.refiner = Encoder(dim=512, depth=1, heads=8, rotary_pos_emb=True, attn_flash=True)
        self.head_bias = nn.Parameter(torch.zeros(vocab_size))
    def forward(self, x):
        w, p = self.rose(x); p_sm = self.particle_down(p); w_raw = self.wave_down(torch.cat([w.real, w.imag], -1))
        w_sm = torch.complex(w_raw[...,:256], w_raw[...,256:])
        p_path = self.stream_rate(p_sm + self.fnet_pos(torch.arange(x.shape[1], device=x.device)))
        w_path = self.phase_bridge(self.stream_phase(w_sm))
        ctx = self.fusion_norm(self.fusion_proj(torch.cat([p_path, w_path], -1)))
        return F.linear(self.refiner(ctx), self.rose.raw_embedding.weight, self.head_bias)

# ==========================================
# 2. ANALYSIS LOGIC
# ==========================================
def smart_load(repo_id, name, cls):
    # Init Model
    model = cls(vocab_size=MAX_VOCAB_SIZE).to(DEVICE)
    print(f"⬇️  Downloading weights for {name}...")
    try: path = hf_hub_download(repo_id, "best.pt")
    except: path = hf_hub_download(repo_id, "pytorch_model.bin")

    state_dict = torch.load(path, map_location="cpu")
    if 'model' in state_dict: state_dict = state_dict['model']
    clean = {k.replace("module.", ""): v for k, v in state_dict.items()}

    # FIXES for Baseline/FNet
    if name == "Baseline":
        new_d = {}
        for k, v in clean.items():
            nk = k if k.startswith("model.") else "model." + k
            if "token_emb.weight" in nk and "emb" not in nk: nk = nk.replace("token_emb.weight", "token_emb.emb.weight")
            new_d[nk] = v
        clean = new_d
    elif name == "FNet":
        new_d = {}
        for k, v in clean.items():
            nk = k.replace("model.", "")
            new_d[nk] = v
        clean = new_d

    model.load_state_dict(clean, strict=False)
    print(f"✅  {name} Ready.")
    return model

def extract_phasor(model, name, token_id):
    with torch.no_grad():
        token_tensor = torch.tensor([token_id], device=DEVICE)
        if name in ["PRISM", "PILLARS"]:
            real_emb = model.rose.raw_embedding(token_tensor)
            params = model.rose.adapter(real_emb)
            D = real_emb.shape[-1]
            z = torch.complex(params[...,:D], params[...,D:])
            return z.squeeze(0).cpu()
        elif name == "FNet":
            x = model.token_emb(token_tensor)
            return torch.complex(x, torch.zeros_like(x)).squeeze(0).cpu()
    return None

def calculate_coherence_dynamic(model, name, id_a, id_b):
    za = extract_phasor(model, name, id_a)
    zb = extract_phasor(model, name, id_b)
    diff = torch.angle(za) - torch.angle(zb)
    weights = torch.abs(za) * torch.abs(zb)

    diff_np = diff.numpy()
    weights_np = weights.numpy()
    weighted_complex_diffs = weights_np * np.exp(1j * diff_np)
    mean_vector = np.sum(weighted_complex_diffs) / (np.sum(weights_np) + 1e-9)
    return np.abs(mean_vector), np.angle(mean_vector), diff_np, weights_np

# ==========================================
# 3. ROBUST CANDIDATE LIST (N = 135)
# ==========================================
candidates_raw = [
    # --- SYNONYMS (Positive Correlation) ---
    ("fast", "quick", "Synonym"), ("big", "large", "Synonym"), ("small", "little", "Synonym"),
    ("start", "begin", "Synonym"), ("end", "finish", "Synonym"), ("smart", "clever", "Synonym"),
    ("hard", "tough", "Synonym"), ("simple", "easy", "Synonym"), ("happy", "glad", "Synonym"),
    ("sad", "unhappy", "Synonym"), ("angry", "mad", "Synonym"), ("correct", "right", "Synonym"),
    ("wrong", "incorrect", "Synonym"), ("shut", "close", "Synonym"), ("buy", "purchase", "Synonym"),
    ("choose", "select", "Synonym"), ("gift", "present", "Synonym"), ("job", "work", "Synonym"),
    ("trip", "journey", "Synonym"), ("lady", "woman", "Synonym"), ("guy", "man", "Synonym"),
    ("street", "road", "Synonym"), ("stone", "rock", "Synonym"), ("speak", "talk", "Synonym"),
    ("listen", "hear", "Synonym"), ("look", "see", "Synonym"), ("run", "sprint", "Synonym"),
    ("jump", "leap", "Synonym"), ("scary", "afraid", "Synonym"), ("rich", "wealthy", "Synonym"),
    ("weird", "strange", "Synonym"), ("quiet", "silent", "Synonym"), ("loud", "noisy", "Synonym"),
    ("trash", "garbage", "Synonym"), ("sick", "ill", "Synonym"), ("thin", "slim", "Synonym"),
    ("near", "close", "Synonym"), ("far", "distant", "Synonym"), ("safe", "secure", "Synonym"),
    ("fix", "repair", "Synonym"), ("mix", "blend", "Synonym"), ("keep", "hold", "Synonym"),
    ("push", "shove", "Synonym"), ("pull", "drag", "Synonym"), ("under", "below", "Synonym"),
    ("above", "over", "Synonym"), ("center", "middle", "Synonym"), ("area", "zone", "Synonym"),

    # --- ANTONYMS (Negative Correlation / Phase Shift) ---
    ("good", "bad", "Antonym"), ("hot", "cold", "Antonym"), ("high", "low", "Antonym"),
    ("up", "down", "Antonym"), ("left", "right", "Antonym"), ("in", "out", "Antonym"),
    ("black", "white", "Antonym"), ("day", "night", "Antonym"), ("sun", "moon", "Antonym"),
    ("boy", "girl", "Antonym"), ("man", "woman", "Antonym"), ("king", "queen", "Antonym"),
    ("life", "death", "Antonym"), ("war", "peace", "Antonym"), ("win", "lose", "Antonym"),
    ("rich", "poor", "Antonym"), ("strong", "weak", "Antonym"), ("hard", "soft", "Antonym"),
    ("loud", "quiet", "Antonym"), ("wet", "dry", "Antonym"), ("clean", "dirty", "Antonym"),
    ("happy", "sad", "Antonym"), ("full", "empty", "Antonym"), ("open", "close", "Antonym"),
    ("first", "last", "Antonym"), ("young", "old", "Antonym"), ("new", "old", "Antonym"),
    ("fast", "slow", "Antonym"), ("tall", "short", "Antonym"), ("heavy", "light", "Antonym"),
    ("dark", "light", "Antonym"), ("true", "false", "Antonym"), ("yes", "no", "Antonym"),
    ("on", "off", "Antonym"), ("top", "bottom", "Antonym"), ("friend", "enemy", "Antonym"),
    ("give", "take", "Antonym"), ("come", "go", "Antonym"), ("rise", "fall", "Antonym"),
    ("north", "south", "Antonym"), ("east", "west", "Antonym"), ("buy", "sell", "Antonym"),
    ("love", "hate", "Antonym"), ("win", "fail", "Antonym"), ("start", "stop", "Antonym"),

    # --- RANDOM (Noise Floor) ---
    ("apple", "car", "Random"), ("banana", "sky", "Random"), ("bread", "cloud", "Random"),
    ("cheese", "door", "Random"), ("milk", "shoe", "Random"), ("water", "book", "Random"),
    ("coffee", "tree", "Random"), ("sugar", "phone", "Random"), ("salt", "idea", "Random"),
    ("meat", "ghost", "Random"), ("soup", "math", "Random"), ("cake", "song", "Random"),
    ("pie", "fish", "Random"), ("egg", "wall", "Random"), ("rice", "nose", "Random"),
    ("tea", "frog", "Random"), ("juice", "star", "Random"), ("fruit", "chair", "Random"),
    ("lemon", "fear", "Random"), ("melon", "bell", "Random"), ("berry", "law", "Random"),
    ("grape", "dog", "Random"), ("plum", "cat", "Random"), ("pear", "bird", "Random"),
    ("lime", "rock", "Random"), ("kiwi", "mud", "Random"), ("bean", "joy", "Random"),
    ("corn", "ice", "Random"), ("nut", "wind", "Random"), ("fig", "pen", "Random"),
    ("yam", "bus", "Random"), ("beef", "sun", "Random"), ("pork", "hat", "Random"),
    ("lamb", "ink", "Random"), ("duck", "map", "Random"), ("goat", "art", "Random"),
    ("cow", "box", "Random"), ("pig", "oil", "Random"), ("hen", "gas", "Random"),
    ("fox", "cup", "Random"), ("wolf", "key", "Random"), ("ant", "bed", "Random"),
    ("bee", "rug", "Random"), ("fly", "mud", "Random"), ("worm", "sky", "Random")
]

# ==========================================
# 4. EXECUTION WITH YOUR CUSTOM TOKENIZER
# ==========================================
print(f"🔑 Loading Tokenizer from {TOKENIZER_ID}...")
try:
    tokenizer = RobertaTokenizerFast.from_pretrained(TOKENIZER_ID)
except:
    print("⚠️  Fallback to base tokenizer if custom fails (Should not happen)")
    tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base")

valid_pairs = []
print(f"🔎 Validating {len(candidates_raw)} candidate pairs...")

# Adding a space prefix " " is standard for RoBERTa tokenizers if words are start of sentence
# but we check both raw and space-prefixed to be safe.
for w1, w2, ptype in candidates_raw:
    # Try with space prefix which RoBERTa often uses for words
    ids1 = tokenizer.encode(" " + w1, add_special_tokens=False)
    ids2 = tokenizer.encode(" " + w2, add_special_tokens=False)

    # Fallback to raw if space fails
    if len(ids1) != 1: ids1 = tokenizer.encode(w1, add_special_tokens=False)
    if len(ids2) != 1: ids2 = tokenizer.encode(w2, add_special_tokens=False)

    if len(ids1) == 1 and len(ids2) == 1:
        id1, id2 = ids1[0], ids2[0]
        if id1 < MAX_VOCAB_SIZE and id2 < MAX_VOCAB_SIZE:
            valid_pairs.append((w1, w2, ptype, id1, id2))

print(f"✅ Found {len(valid_pairs)} valid single-token pairs for this tokenizer.")

MODELS_TO_TEST = [
    ("PRISM", "prism-lab/prism-v2-wikitext", LocalPRISM),
    ("PILLARS", "prism-lab/pillars-compact-wikitext", LocalPillars),
    ("FNet", "prism-lab/hybrid-fnet-prism-custom", HybridFNetMLM)
]

all_results = {}

for name, repo, cls in MODELS_TO_TEST:
    print(f"\n🧪  Analyzing {name}...")
    try:
        model = smart_load(repo, name, cls)
        model.eval()

        results = []
        for w1, w2, ptype, id1, id2 in valid_pairs:
            R, angle, diffs, weights = calculate_coherence_dynamic(model, name, id1, id2)
            results.append({"Pair": f"{w1}-{w2}", "Type": ptype, "R": R, "Diffs": diffs, "Weights": weights})

        all_results[name] = results
        df = pd.DataFrame(results)
        print(f"📊 {name} Results (Mean R):")
        if not df.empty:
            print(df.groupby("Type")["R"].mean())
        del model; torch.cuda.empty_cache(); gc.collect()
    except Exception as e:
        print(f"❌ {name} Failed: {e}")

# Plotting
if len(all_results) > 0:
    fig = plt.figure(figsize=(12, 10))
    cols = ["Synonym", "Antonym", "Random"]
    rows = list(all_results.keys())
    colors = {"Synonym": "red", "Antonym": "purple", "Random": "gray"}

    idx = 1
    for model_name in rows:
        data = all_results[model_name]
        df = pd.DataFrame(data)
        if df.empty: continue

        best_syn = df[df["Type"]=="Synonym"].sort_values("R", ascending=False).iloc[0]
        best_ant = df[df["Type"]=="Antonym"].sort_values("R", ascending=False).iloc[0]
        best_rnd = df[df["Type"]=="Random"].sort_values("R", ascending=True).iloc[0]

        examples = [best_syn, best_ant, best_rnd]
        for i, ex in enumerate(examples):
            ax = fig.add_subplot(len(rows), 3, idx, projection='polar')
            c = colors[ex["Type"]]
            ax.hist(ex["Diffs"], bins=30, weights=ex["Weights"], color=c, alpha=0.6, density=True)
            ax.annotate("", xy=(0, ex["R"]), xytext=(0,0), arrowprops=dict(facecolor='black', width=1.5, headwidth=8, alpha=0.9))

            label = f"{ex['Pair']}\nR={ex['R']:.3f}"
            if i == 1: ax.set_title(f"Model: {model_name}\n{label}", fontsize=10, weight='bold')
            else: ax.set_title(label, fontsize=9)

            ax.set_yticklabels([]); ax.set_xticklabels([])
            idx += 1

    plt.tight_layout()
    plt.savefig("multi_model_compass_publication.png", dpi=300)
    print("\n📸 Saved plot to 'multi_model_compass_publication.png'")

In [ ]:
# ========================
# 6. ANGULAR TOPOLOGY PLOT
# ========================
import seaborn as sns

def plot_angular_topology(all_results):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

    # We only care about Synonyms to see "How" they align
    models = ["FNet", "PRISM"]
    colors = {"FNet": "blue", "PRISM": "red"}

    for i, name in enumerate(models):
        if name not in all_results: continue

        # Collect ALL phase differences for Synonyms across all pairs
        # We flatten the list of angles
        angles = []
        data = all_results[name]
        for item in data:
            if item["Type"] == "Synonym":
                # Convert radians to degrees for readability
                deg = np.degrees(item["Diffs"])
                # Wrap to -180 to 180
                deg = (deg + 180) % 360 - 180
                angles.extend(deg)

        sns.histplot(angles, ax=axes[i], bins=60, color=colors[name], stat="density", kde=True)
        axes[i].set_title(f"{name} Phase Topology (Synonyms)")
        axes[i].set_xlabel("Phase Difference (Degrees)")
        axes[i].set_xlim(-180, 180)
        axes[i].grid(True, alpha=0.3)

        # Add annotation
        if name == "FNet":
            axes[i].text(0, 0.01, "BINARY\n(Sign Flips)", ha='center', color='black', fontweight='bold')
        else:
            axes[i].text(0, 0.01, "CONTINUOUS\n(Rotation)", ha='center', color='black', fontweight='bold')

    plt.tight_layout()
    plt.savefig("angular_topology_comparison.png", dpi=300)
    print("📸 Saved topology proof to 'angular_topology_comparison.png'")

# Run the plot with your existing results
plot_angular_topology(all_results)

In [ ]:
# ==========================================
# 7. RATE VS PHASE DISSOCIATION PROBE
# ==========================================
from scipy.stats import pearsonr

def check_cosine_and_magnitude(model, name, id_a, id_b):
    z_a = extract_phasor(model, name, id_a)
    z_b = extract_phasor(model, name, id_b)

    # --- 1. Vector Cosine Similarity (The Standard Metric) ---
    # For Complex (PRISM), we treat Re/Im as two coordinate dimensions
    if name in ["PRISM", "PILLARS"]:
        # Flatten: [Re_1, Im_1, Re_2, Im_2, ...]
        vec_a = torch.cat([z_a.real, z_a.imag], -1)
        vec_b = torch.cat([z_b.real, z_b.imag], -1)
    else:
        # FNet is already real
        vec_a = z_a.real
        vec_b = z_b.real

    vec_sim = F.cosine_similarity(vec_a.unsqueeze(0), vec_b.unsqueeze(0)).item()

    # --- 2. Magnitude Correlation (The "Rate Coding" Check) ---
    # Do these words emphasize the same dimensions?
    mag_a = torch.abs(z_a).numpy()
    mag_b = torch.abs(z_b).numpy()

    # Pearson correlation of the magnitude profiles
    # If the model uses Rate Coding, this should be HIGH.
    # If the model is Iso-Energetic (PRISM), this should be NOISE.
    if np.std(mag_a) < 1e-6 or np.std(mag_b) < 1e-6:
        mag_corr = 0.0 # Handle constant magnitude case
    else:
        mag_corr, _ = pearsonr(mag_a, mag_b)

    return vec_sim, mag_corr

print("\n⚖️  Running Rate vs. Phase Dissociation...")

comparison_data = []

# We only check Synonyms to see how they agree
synonym_pairs = [p for p in valid_pairs if p[2] == "Synonym"]

for name, repo, cls in MODELS_TO_TEST:
    try:
        model = smart_load(repo, name, cls)
        model.eval()

        vec_scores = []
        mag_scores = []

        for w1, w2, _, id1, id2 in synonym_pairs:
            v_sim, m_corr = check_cosine_and_magnitude(model, name, id1, id2)
            vec_scores.append(v_sim)
            mag_scores.append(m_corr)

        avg_vec = np.mean(vec_scores)
        avg_mag = np.mean(mag_scores)

        comparison_data.append({
            "Model": name,
            "Vector Sim (Direction)": avg_vec,
            "Mag Corr (Loudness)": avg_mag
        })

        del model; torch.cuda.empty_cache()
    except Exception as e:
        print(f"Skipping {name}: {e}")

# Display the "Dissociation" Table
df_comp = pd.DataFrame(comparison_data)
print("\n🔥 THE DISSOCIATION TABLE 🔥")
print(df_comp.set_index("Model"))

In [ ]:
import pandas as pd

# 1. Convert the valid_pairs list to a DataFrame
df_stats = pd.DataFrame(valid_pairs, columns=["Word1", "Word2", "Category", "ID1", "ID2"])

# 2. Print the statistics
print("\n📊 DATASET STATISTICS (Post-Filtering)")
print("========================================")
# Counts per category
counts = df_stats["Category"].value_counts()
print(counts)

print("----------------------------------------")
print(f"✅ Total Valid Pairs: {len(df_stats)}")
print("========================================")

# 3. Helper for your Paper's Table
print("\n📝 UPDATE FOR TABLE 2 (Count Column):")
for category, count in counts.items():
    print(f"   > {category}: {count}")